# Step 5: Data Cleaning & Preprocessing
Cart2Insights — issues found in Step 4 and how each is resolved.

In [ ]:
import pandas as pd
import numpy as np

RAW = '../data/raw/'
CLEAN = '../data/cleaned/'
import os
os.makedirs(CLEAN, exist_ok=True)

customers = pd.read_csv(RAW + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(RAW + 'olist_orders_dataset.csv')
products = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(RAW + 'product_category_name_translation.csv')

## 1. Orders — parse dates, handle missing delivery timestamps
`order_approved_at` (160 nulls), `order_delivered_carrier_date` (1,783 nulls), and
`order_delivered_customer_date` (2,965 nulls) are missing almost entirely because those
orders were never completed (canceled/unavailable/processing) — this is expected, not an error.
We keep the nulls (don't impute fake delivery dates) and instead filter on them explicitly
wherever delivery-time metrics are computed.

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Sanity check: confirm missing delivery dates correlate with non-delivered status
print(orders.loc[orders['order_delivered_customer_date'].isnull(), 'order_status'].value_counts())

## 2. Reviews — drop duplicate review_id rows
814 duplicate `review_id` rows exist (same review recorded more than once, sometimes
with a resent answer timestamp). Keep the most recent `review_answer_timestamp` per review_id.

In [ ]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

before = len(reviews)
reviews = (reviews.sort_values('review_answer_timestamp')
                   .drop_duplicates(subset='review_id', keep='last'))
print(f'Dropped {before - len(reviews)} duplicate review_id rows')

# review_comment_title / review_comment_message nulls are legitimate (no comment left) —
# fill with empty string rather than dropping, since review_score is still valid
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

## 3. Geolocation — drop duplicate zip/lat/lng rows
261,831 fully duplicate rows (same zip code sampled multiple times at identical coordinates).

In [ ]:
before = len(geolocation)
geolocation = geolocation.drop_duplicates()
print(f'Dropped {before - len(geolocation)} duplicate geolocation rows')

# Collapse to one representative lat/lng per zip prefix (median, robust to outlier pings)
geolocation_agg = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'median'),
    geolocation_lng=('geolocation_lng', 'median'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

## 4. Products — handle missing category & dimensions
610 products have no category and no name/description length (likely delisted products).
2 products are missing weight/dimensions. Given these are a small fraction (~1.8%) of
32,951 products, we label missing categories as 'unknown' rather than dropping the rows
(dropping would lose valid order_items/sales history tied to these products).

In [ ]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Impute missing physical dimensions with median (only 2 rows affected)
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].median())

## 5. Payments — standardize payment_type
3 rows have `payment_type = 'not_defined'`. Kept as its own category rather than dropped,
since the order/payment amount itself is still valid — only the method label is unknown.

In [ ]:
print('Payment types:', payments['payment_type'].unique())
# No change needed — 'not_defined' is a legitimate (if rare) category, not an error

## 6. Primary key uniqueness — final verification

In [ ]:
checks = {
    'orders.order_id': orders['order_id'].is_unique,
    'customers.customer_id': customers['customer_id'].is_unique,
    'products.product_id': products['product_id'].is_unique,
    'sellers.seller_id': sellers['seller_id'].is_unique,
    'reviews.review_id (post-dedup)': reviews['review_id'].is_unique,
}
for k, v in checks.items():
    print(f'{k}: {"VALID" if v else "INVALID"}')

## 7. Save cleaned tables

In [ ]:
customers.to_csv(CLEAN + 'customers.csv', index=False)
geolocation_agg.to_csv(CLEAN + 'geolocation.csv', index=False)
order_items.to_csv(CLEAN + 'order_items.csv', index=False)
payments.to_csv(CLEAN + 'order_payments.csv', index=False)
reviews.to_csv(CLEAN + 'order_reviews.csv', index=False)
orders.to_csv(CLEAN + 'orders.csv', index=False)
products.to_csv(CLEAN + 'products.csv', index=False)
sellers.to_csv(CLEAN + 'sellers.csv', index=False)
category_translation.to_csv(CLEAN + 'category_translation.csv', index=False)
print('All cleaned tables saved to data/cleaned/')